In [0]:
from pyspark.sql.functions import col, countDistinct, isnan, isnull
from pyspark.sql.types import DoubleType, FloatType

silver_schemapath = "subscription_pipeline.silver"

In [0]:
silver_tables = [
    "employees",
    "customers",
    "countries",
    "fx_rates",
    "products",
    "opportunity"
]

def cleaning_report(table_name):
    df = spark.read.table(f"{silver_schemapath}.{table_name}")
    total_rows = df.count()
    report = []
    for c in df.columns:
        col_type = df.schema[c].dataType
        if isinstance(col_type, (DoubleType, FloatType)):
            null_count = df.filter(isnull(col(c)) | isnan(col(c))).count()
        else:
            null_count = df.filter(isnull(col(c))).count()
        distinct_count = df.select(countDistinct(col(c))).collect()[0][0]
        report.append({
            "table": table_name,
            "column": c,
            "total_rows": total_rows,
            "null_count": null_count,
            "distinct_count": distinct_count
        })
    return report

all_reports = []
for t in silver_tables:
    print(f"Analyzing {t}...")
    all_reports.extend(cleaning_report(t))

report_df = spark.createDataFrame(all_reports)
print("\n✓ Data Quality Report Complete")
display(report_df)